# K-fold per video: distribuzione delle classi per fold

Notebook configurabile per **verificare lo schema di cross-validation per video** (`src/datasets/kfold.py`) e
documentare la distribuzione delle classi in ogni fold.

Per ogni fold e per ogni insieme (train / validation, e test se `TEST_RATIO > 0`) il notebook calcola, **sull'intero
dataset e senza sampling**:
- **frame** che contengono la classe (almeno un pixel) e percentuale sui frame dell'insieme;
- **pixel** della classe e percentuale sui pixel validi dell'insieme;
- **video** che contengono la classe;
- se la classe è **valutabile** nel fold (presente sia in train sia in validation); altrimenti la metrica è N/A.

Regole dello schema (vedi `docs/metrics_convention.md`):
1. si divide per video, mai per frame;
2. una classe presente in **un solo video** non può essere allenata e validata: il video che la contiene è
   **bloccato in train** e la classe non è valutata (N/A);
3. una classe in più video compare in validation in `min(K, video con la classe)` fold, quando possibile;
4. i fold dipendono solo dagli id dei video e dal seed, non dall'ordine del dataset;
5. a parità di copertura, i fold sono **bilanciati**: la quota di pixel delle classi più popolose in ogni validation
   deve restare vicina a quella del dataset (sezione "Bilanciamento della distribuzione").

Ogni maschera viene letta **una sola volta** (i conteggi sono messi in cache su disco, `USE_CACHE`).
In fondo al notebook viene generato un **report PDF**.

# CONFIGURAZIONE PRINCIPALE

In [ ]:
# Change ONLY this
#   "baseline/cholec_seg8k"   (K-fold: è il dataset per cui questo schema è usato)
#   "baseline/dsad"           (split ufficiale nel training: qui solo per esplorare un k-fold ipotetico)
#   "baseline/sarrarp50"      (idem)
DATASET_CONFIG = "baseline/cholec_seg8k"

# Parametri dello schema (di default gli stessi dei config: data.split.*)
N_FOLDS = 4
SEED = 42
TEST_RATIO = 0.0          # frazione di video tenuta fuori come test PRIMA del k-fold (0 = nessun test)
GROUP_KEY = "video"       # DSAD usa automaticamente "surgery_id"

# Bilanciamento della distribuzione tra i fold (stessi parametri di data.split nei YAML)
BALANCE = True
N_RESTARTS = 30
BALANCE_MIN_PIXEL_PCT = 1.0   # le classi sotto questa % di pixel non entrano nel bilanciamento

# Cache dei conteggi per frame (evita di rileggere tutte le maschere ai rilanci)
USE_CACHE = True

# Cartella di output (relativa alla root del progetto): CSV, folds.json e PDF
REPORT_DIR = "docs/reports/kfold_validation"

# Nome del config principale Hydra del progetto.
HYDRA_CONFIG_NAME = "config"

PROJECT_ROOT = None

In [ ]:
# ============================================================
# SETUP PROGETTO
# ============================================================

from pathlib import Path
import sys


def find_project_root(start=None):
    """Find the repository root containing configs/ and src/."""
    start = Path(start or Path.cwd()).resolve()

    for path in [start, *start.parents]:
        if (path / "src").is_dir() and (path / "configs").is_dir():
            return path

    raise FileNotFoundError(
        "Impossibile trovare la root della repo. "
        "Imposta PROJECT_ROOT manualmente nella cella di configurazione."
    )


if PROJECT_ROOT is None:
    PROJECT_ROOT = find_project_root()
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# IMPORT
# ============================================================

import json
import pickle
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

from src.datasets.validation import (
    get_samples,
    get_class_names,
    compute_sample_counts,
    kfold_class_distribution,
    kfold_distribution_deviation,
)
from src.datasets.kfold import build_kfold_plan, plan_kfold, kfold_indices

In [ ]:
# ============================================================
# CARICAMENTO CONFIG + DATASET
# ============================================================

with initialize_config_dir(version_base=None, config_dir=str(PROJECT_ROOT / "configs")):
    cfg = compose(config_name=HYDRA_CONFIG_NAME, overrides=[f"dataset={DATASET_CONFIG}"])

dataset = instantiate(cfg.data.dataset)
DATASET_NAME = str(cfg.data.name)
class_names = get_class_names(dataset)

print(f"Dataset: {DATASET_NAME} ({type(dataset).__name__})")
print(f"Frame  : {len(dataset.samples)}")
print(f"Classi : {class_names}")
print(f"Split nel config del dataset: {OmegaConf.to_container(cfg.data.split)}")

## Conteggi per frame (una sola lettura delle maschere)

`compute_sample_counts` legge ogni maschera una volta e restituisce, per ogni frame, i pixel di ogni classe.
Da questi conteggi si ricavano **tutto**: la presenza delle classi per video (usata dallo schema dei fold) e le
tabelle per fold. Con `USE_CACHE` vengono salvati in `notebooks/outputs/` e riusati ai rilanci
(cancella il file se cambi le maschere del dataset).

In [ ]:
# ============================================================
# CONTEGGI PER FRAME (con cache)
# ============================================================

samples = get_samples(dataset)
all_indices = list(range(len(samples)))

cache_dir = PROJECT_ROOT / "notebooks" / "outputs"
cache_dir.mkdir(parents=True, exist_ok=True)
cache_file = cache_dir / f"kfold_cache_{DATASET_NAME}.pkl"

sample_cache = None
if USE_CACHE and cache_file.exists():
    with open(cache_file, "rb") as f:
        stored = pickle.load(f)
    if stored["n_samples"] == len(samples) and stored["class_names"] == class_names:
        sample_cache = stored["cache"]
        print(f"Cache caricata: {cache_file.name}")
    else:
        print("Cache non compatibile con il dataset: ricalcolo.")

if sample_cache is None:
    start = time.time()
    sample_cache = compute_sample_counts(dataset, all_indices)
    print(f"Maschere lette in {time.time() - start:.0f} s")
    if USE_CACHE:
        with open(cache_file, "wb") as f:
            pickle.dump({"n_samples": len(samples), "class_names": class_names, "cache": sample_cache}, f)
        print(f"Cache salvata: {cache_file.name}")

print(f"Frame: {len(sample_cache)} | pixel ignorati (etichette fuori range): "
      f"{sum(c[1] for c in sample_cache.values())}")

## Piano dei fold

`build_kfold_plan` costruisce lo schema (stesse funzioni usate da `start_experiment.py`):
1. i video che contengono classi presenti in **un solo video** sono bloccati in train;
2. i restanti video sono assegnati ai fold con stratificazione multilabel sulla presenza delle classi;
3. uno scambio greedy tra fold massimizza il numero di fold in cui ogni classe è valutabile (criterio principale) e,
   a parità, avvicina la quota di pixel delle classi popolose in ogni validation a quella del dataset (criterio
   secondario); vengono provati più tentativi iniziali deterministici (`seed`, `seed+1`, ...) e si tiene il migliore.

In [ ]:
# ============================================================
# PIANO DEI FOLD
# ============================================================

split_config = {
    "group_by": GROUP_KEY,
    "n_folds": N_FOLDS,
    "test_ratio": TEST_RATIO,
    "seed": SEED,
    "balance": BALANCE,
    "n_restarts": N_RESTARTS,
    "balance_min_pixel_pct": BALANCE_MIN_PIXEL_PCT,
}
plan, groups = build_kfold_plan(dataset, split_config, cache=sample_cache)

pinned = {
    plan.group_ids[g]: [plan.class_names[c] for c in classes]
    for g, classes in plan.pinned_reasons.items()
}
print(f"Video: {len(plan.group_ids)} | fold: {plan.n_folds} | seed: {plan.seed} | test_ratio: {plan.test_ratio}")
print("Video bloccati in train (contengono classi presenti in un solo video):", pinned or "nessuno")
if plan.test:
    print("Video di test:", [plan.group_ids[g] for g in plan.test])

fold_table = plan.fold_table()
display(fold_table)

In [ ]:
# ============================================================
# COPERTURA DELLE CLASSI NEI FOLD
# ============================================================
# ok                  = la classe è in train E in validation: valutabile
# n/a (not in val)    = la classe non è nella validation del fold
# n/a (not in train)  = la classe è in validation ma non in train: il valore 0 non misurerebbe il modello

coverage = plan.coverage_table()


def color_status(value):
    if isinstance(value, str) and value.startswith("n/a"):
        return "background-color: #fde2e2"
    return ""


shown = coverage.drop(columns=["valid_fold_ids"])
styler = shown.style
styler = (styler.map if hasattr(styler, "map") else styler.applymap)(
    color_status, subset=[c for c in shown.columns if c.startswith("fold")]
)
display(styler)

never = coverage.loc[coverage["valid_folds"] == 0, "class"].tolist()
partial = coverage.loc[(coverage["valid_folds"] > 0) & (coverage["valid_folds"] < plan.n_folds), "class"].tolist()
print(f"Valutabili in tutti i fold : {coverage.loc[coverage['valid_folds'] == plan.n_folds, 'class'].tolist()}")
print(f"Valutabili solo in alcuni  : {partial}")
print(f"Mai valutabili (N/A)       : {never}")

## Distribuzione delle classi per fold

Per ogni fold, in train e in validation: **frame**, **pixel** e **video** che contengono la classe.
`frame_percentage` e `pixel_percentage` sono calcolati sull'insieme (train o validation) del fold.

In [ ]:
# ============================================================
# DISTRIBUZIONE PER FOLD: frame, pixel, video per classe
# ============================================================

dist = kfold_class_distribution(dataset, plan, group_key=GROUP_KEY, cache=sample_cache)


def wide(value):
    """Classe x (fold, insieme) per una colonna di `dist`."""
    table = dist.pivot_table(index=["class_id", "class"], columns=["fold", "split"], values=value,
                             aggfunc="sum", sort=False)
    return table.droplevel("class_id")


for value, title, fmt in [
    ("frames", "Frame che contengono la classe", "{:,.0f}"),
    ("frame_percentage", "Frame che contengono la classe (% dei frame dell'insieme)", "{:.1f}"),
    ("pixels", "Pixel della classe", "{:,.0f}"),
    ("pixel_percentage", "Pixel della classe (% dei pixel dell'insieme)", "{:.3f}"),
    ("videos", "Video che contengono la classe", "{:,.0f}"),
]:
    print(title)
    display(wide(value).style.format(fmt))

print("Dimensione degli insiemi")
display(dist.drop_duplicates(["fold", "split"])[["fold", "split", "total_videos", "total_frames", "valid_pixels"]]
        .reset_index(drop=True))

## Bilanciamento della distribuzione

Le classi più popolose devono avere in ogni validation una quota di pixel simile a quella del dataset. Il riferimento
(`pool_pct`) è la quota sui video che possono essere validati (tutti i video tranne quelli bloccati in train e il test):
i fold di validation partizionano proprio quell'insieme, quindi la media delle quote dei fold è il riferimento.

- `rel_dev` = `100 * (val_pct - pool_pct) / pool_pct`: 0 significa la stessa quota del riferimento;
- `populous` = classi con almeno `BALANCE_MIN_PIXEL_PCT` % dei pixel del riferimento.

Con pochi video lo scostamento non può azzerarsi: ogni video ha una composizione diversa.

In [ ]:
# ============================================================
# BILANCIAMENTO DELLA DISTRIBUZIONE TRA I FOLD
# ============================================================

deviation = kfold_distribution_deviation(dist, BALANCE_MIN_PIXEL_PCT)
populous = deviation[deviation["populous"]]

print(f"Classi popolose (>= {BALANCE_MIN_PIXEL_PCT}% dei pixel): {populous['class'].unique().tolist()}")

print("\nQuota di pixel (%) in validation per fold, con il riferimento")
shares = populous.pivot_table(index="class", columns="fold", values="val_pct", sort=False)
shares.insert(0, "riferimento", populous.drop_duplicates("class").set_index("class")["pool_pct"])
display(shares.style.format("{:.2f}"))

print("Scostamento relativo (%) rispetto al riferimento")
relative = populous.pivot_table(index="class", columns="fold", values="rel_dev", sort=False)
display(relative.style.format("{:+.0f}").background_gradient(cmap="RdBu_r", vmin=-100, vmax=100))

abs_rel = populous["rel_dev"].abs()
print(f"Scostamento medio |rel_dev| = {abs_rel.mean():.1f}%   massimo = {abs_rel.max():.1f}%")
print("Massimo per classe:", populous.groupby("class")["rel_dev"].apply(lambda x: round(x.abs().max())).to_dict())

fold_sizes = dist[dist["split"] == "validation"].drop_duplicates("fold")[["fold", "total_videos", "total_frames"]]
print("\nDimensione delle validation (frame): min", int(fold_sizes["total_frames"].min()),
      "| max", int(fold_sizes["total_frames"].max()))

## Verifiche

Controlli automatici sullo schema e sulle tabelle. Se un controllo fallisce, il notebook si ferma con un errore.

In [ ]:
# ============================================================
# VERIFICHE
# ============================================================

checks = []


def check(name, ok):
    checks.append((name, bool(ok)))
    print(("OK    " if ok else "FAIL  ") + name)


all_videos = set(groups)
test_videos = {plan.group_ids[g] for g in plan.test}
pinned_videos = set(pinned)
total_frames = np.sum([c[0] > 0 for c in sample_cache.values()], axis=0)
total_pixels = np.sum([c[0] for c in sample_cache.values()], axis=0)

validated = []
for k in range(plan.n_folds):
    g = plan.groups_for(k)
    tr, va, te = set(g["train"]), set(g["validation"]), set(g["test"])
    check(f"fold {k}: train / validation / test disgiunti", not (tr & va) and not (tr & te) and not (va & te))
    check(f"fold {k}: l'unione copre tutti i video", tr | va | te == all_videos)
    validated += g["validation"]

    idx = kfold_indices(plan, groups, k)
    check(f"fold {k}: nessun frame in comune tra train e validation", not set(idx["train"]) & set(idx["validation"]))
    check(f"fold {k}: i frame sono tutti assegnati",
          len(idx["train"]) + len(idx["validation"]) + len(idx["test"]) == len(samples))

    part = dist[dist["fold"] == k]
    for split_name, part_split in part.groupby("split"):
        check(f"fold {k}/{split_name}: somma delle percentuali di pixel = 100",
              abs(part_split["pixel_percentage"].sum() - 100) < 1e-6)
    sums = part.groupby("class_id")[["frames", "pixels"]].sum()
    check(f"fold {k}: frame per classe (train+val+test) = frame del dataset", sums["frames"].tolist() == total_frames.tolist())
    check(f"fold {k}: pixel per classe (train+val+test) = pixel del dataset", sums["pixels"].tolist() == total_pixels.tolist())

    tr_f = part[part["split"] == "train"].set_index("class_id")["frames"]
    va_f = part[part["split"] == "validation"].set_index("class_id")["frames"]
    expected = ((tr_f > 0) & (va_f > 0)).to_numpy()
    check(f"fold {k}: 'valutabile' = presente in train e in validation", (expected == plan.evaluable_classes(k)).all())

check("ogni video non bloccato è in validation in un solo fold",
      sorted(validated) == sorted(all_videos - pinned_videos - test_videos))
check("i video bloccati non sono mai in validation", not pinned_videos & set(validated))

video_counts = plan.presence.sum(axis=0)
single = [c for c in range(len(class_names)) if video_counts[c] == 1]
check("le classi presenti in un solo video non sono mai valutabili",
      all(not plan.evaluable_matrix()[:, c].any() for c in single))
check("nessun pixel con etichetta fuori range", all(c[1] == 0 for c in sample_cache.values()))

rng = np.random.default_rng(0)
perm = rng.permutation(len(plan.group_ids))
shuffled = plan_kfold(
    plan.presence[perm], [plan.group_ids[i] for i in perm], plan.class_names,
    n_folds=plan.n_folds, test_ratio=plan.test_ratio, seed=plan.seed,
    group_sizes=plan.group_sizes[perm],
    group_pixels=None if plan.group_pixels is None else plan.group_pixels[perm],
    n_restarts=N_RESTARTS, balance_min_pixel_pct=BALANCE_MIN_PIXEL_PCT,
)
check("riproducibile: stessi fold rimescolando l'ordine dei video",
      all(plan.groups_for(k) == shuffled.groups_for(k) for k in range(plan.n_folds)))

check("la quota di pixel del riferimento somma a 100", abs(deviation.drop_duplicates("class_id")["pool_pct"].sum() - 100) < 1e-6)
check("scostamento calcolato per tutte le classi e tutti i fold", len(deviation) == plan.n_folds * len(class_names))

failed = [name for name, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} controlli superati")
assert not failed, f"Controlli falliti: {failed}"

## Grafici

In [ ]:
# ============================================================
# GRAFICI
# ============================================================

SPLIT_COLORS = {"train": "#4c72b0", "validation": "#dd8452", "test": "#55a868"}
evaluable_matrix = plan.evaluable_matrix()


def make_pixel_figure():
    """Pixel (%) di ogni classe in train e validation, un pannello per fold."""
    cols = 2 if plan.n_folds > 1 else 1
    rows = int(np.ceil(plan.n_folds / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(8.5 * cols, 4.2 * rows), squeeze=False)
    x = np.arange(len(class_names))
    for k in range(plan.n_folds):
        ax = axes[k // cols][k % cols]
        part = dist[dist["fold"] == k]
        splits = [s for s in ("train", "validation", "test") if s in set(part["split"])]
        width = 0.8 / len(splits)
        for i, split_name in enumerate(splits):
            values = part[part["split"] == split_name].sort_values("class_id")["pixel_percentage"].to_numpy()
            ax.bar(x + i * width, values, width, label=split_name, color=SPLIT_COLORS[split_name])
        ax.set_xticks(x + width * (len(splits) - 1) / 2)
        ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
        for tick, ok in zip(ax.get_xticklabels(), evaluable_matrix[k]):
            if not ok:
                tick.set_color("#c0392b")
        ax.set_yscale("log")
        ax.set_ylabel("Pixel (%) - scala log")
        ax.set_title(f"Fold {k}: distribuzione dei pixel")
        ax.legend(fontsize=8)
    for k in range(plan.n_folds, rows * cols):
        axes[k // cols][k % cols].axis("off")
    fig.suptitle(f"{DATASET_NAME}: pixel per classe in train e validation (rosso = classe N/A nel fold)", y=1.0)
    fig.tight_layout()
    return fig


def make_heatmap_figure(value, title, fmt="{:,.0f}"):
    """Heatmap classe x (fold, insieme) con i valori annotati."""
    table = wide(value)
    data = table.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(max(8, 1.0 * table.shape[1] + 3), max(4, 0.45 * table.shape[0] + 1.5)))
    im = ax.imshow(np.log1p(data), aspect="auto", cmap="viridis")
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels([f"f{f}\n{s[:5]}" for f, s in table.columns], fontsize=8)
    ax.set_yticks(range(table.shape[0]))
    ax.set_yticklabels(table.index, fontsize=8)
    peak = np.log1p(data).max() or 1
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            shade = "white" if np.log1p(data[i, j]) < 0.6 * peak else "black"
            ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center", fontsize=7, color=shade)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="log(1 + valore)")
    fig.tight_layout()
    return fig


def make_deviation_figure():
    """Scostamento relativo della quota di pixel in validation, per le classi popolose."""
    table = populous.pivot_table(index="class", columns="fold", values="rel_dev", sort=False)
    fig, ax = plt.subplots(figsize=(max(8, 1.3 * len(table)), 4.5))
    x = np.arange(len(table))
    width = 0.8 / plan.n_folds
    for k in range(plan.n_folds):
        ax.bar(x + k * width, table[k].to_numpy(), width, label=f"fold {k}")
    ax.axhline(0, color="black", lw=0.8)
    for level in (-50, 50):
        ax.axhline(level, color="gray", ls="--", lw=0.8)
    ax.set_xticks(x + width * (plan.n_folds - 1) / 2)
    ax.set_xticklabels(table.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Scostamento relativo (%)")
    ax.set_title(f"{DATASET_NAME}: quota di pixel in validation rispetto al riferimento (classi popolose)")
    ax.legend(fontsize=8, ncol=plan.n_folds)
    fig.tight_layout()
    return fig


fig = make_pixel_figure()
plt.show()
fig = make_deviation_figure()
plt.show()
fig = make_heatmap_figure("frames", f"{DATASET_NAME}: frame che contengono la classe")
plt.show()
fig = make_heatmap_figure("videos", f"{DATASET_NAME}: video che contengono la classe")
plt.show()

## Salvataggio delle tabelle

In [ ]:
# ============================================================
# CSV + folds.json
# ============================================================

out_dir = PROJECT_ROOT / REPORT_DIR
out_dir.mkdir(parents=True, exist_ok=True)


def as_text(frame):
    """Liste -> stringhe, per salvare in CSV."""
    frame = frame.copy()
    for column in frame.columns:
        frame[column] = frame[column].map(
            lambda v: ", ".join(map(str, v)) if isinstance(v, (list, tuple)) else v)
    return frame


dist.to_csv(out_dir / f"{DATASET_NAME}_kfold_class_distribution.csv", index=False)
deviation.to_csv(out_dir / f"{DATASET_NAME}_kfold_distribution_deviation.csv", index=False)
as_text(coverage).to_csv(out_dir / f"{DATASET_NAME}_kfold_coverage.csv", index=False)
as_text(fold_table).to_csv(out_dir / f"{DATASET_NAME}_kfold_fold_table.csv", index=False)
plan.save(out_dir / f"{DATASET_NAME}_kfold_folds.json")

print("Salvati in", out_dir)
for path in sorted(out_dir.glob(f"{DATASET_NAME}_kfold_*")):
    print("  ", path.name)

## Report PDF

Ultima cella: genera `docs/reports/kfold_validation/<dataset>_kfold_report.pdf` con
copertina e parametri, riepilogo, dimensione dei fold, copertura delle classi, **una pagina di tabelle per fold**
(frame, pixel e video per classe in train e validation), grafici ed esito dei controlli.

In [ ]:
# ============================================================
# GENERATE K-FOLD REPORT (PDF)
# ============================================================
#
# Struttura del PDF:
#   1. Copertina e parametri
#   2. Riepilogo (video bloccati, classi valutabili / N/A)
#   3. Dimensione dei fold
#   4. Copertura delle classi nei fold
#   5. Una pagina per fold: frame, pixel e video per classe in train e validation
#   6. Bilanciamento della distribuzione (classi popolose) e grafici
#   7. Esito dei controlli

from matplotlib.backends.backend_pdf import PdfPages

PAGE = (11.69, 8.27)  # A4 landscape
HEADER_COLOR, ALT_COLOR, BAD_COLOR = "#2c3e50", "#f4f6f8", "#fde2e2"
report_path = out_dir / f"{DATASET_NAME}_kfold_report.pdf"


def new_page(title, subtitle=None):
    fig = plt.figure(figsize=PAGE)
    fig.text(0.04, 0.95, title, fontsize=16, fontweight="bold", va="top")
    if subtitle:
        fig.text(0.04, 0.91, subtitle, fontsize=9, color="#555555", va="top")
    return fig


def text_block(fig, lines, top=0.84, size=10.5, step=0.036):
    y = top
    for line in lines:
        bold = line.startswith("**")
        fig.text(0.05, y, line.strip("*"), fontsize=size, fontweight="bold" if bold else "normal", va="top")
        y -= step
    return y


def draw_table(fig, frame, rect, col_widths=None, bad_cells=(), font=8, scale=1.5):
    """frame: DataFrame di sole stringhe. bad_cells: (riga, colonna) da evidenziare."""
    ax = fig.add_axes(rect)
    ax.axis("off")
    table = ax.table(cellText=frame.values.tolist(), colLabels=list(frame.columns),
                     colWidths=col_widths, loc="upper center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(font)
    table.scale(1, scale)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if r == 0:
            cell.set_facecolor(HEADER_COLOR)
            cell.set_text_props(color="white", fontweight="bold")
        elif (r - 1, c) in bad_cells:
            cell.set_facecolor(BAD_COLOR)
        elif r % 2 == 0:
            cell.set_facecolor(ALT_COLOR)
    return table


def n(value):
    return f"{int(value):,}"


generated = datetime.now().strftime("%d/%m/%Y %H:%M")
n_frames = len(samples)

with PdfPages(report_path) as pdf:

    # ---- 1. Copertina ----
    fig = plt.figure(figsize=PAGE)
    fig.text(0.5, 0.68, "Cross-validation per video", fontsize=30, fontweight="bold", ha="center")
    fig.text(0.5, 0.60, f"Distribuzione delle classi per fold: {DATASET_NAME}", fontsize=16, ha="center")
    text_lines = [
        f"Dataset: {DATASET_NAME} ({type(dataset).__name__})",
        f"Video: {len(plan.group_ids)}   |   frame: {n_frames:,}   |   classi: {len(class_names)}",
        f"Fold: K = {plan.n_folds}   |   seed: {plan.seed}   |   test_ratio: {plan.test_ratio}",
        f"Generato il {generated}",
    ]
    for i, line in enumerate(text_lines):
        fig.text(0.5, 0.48 - 0.045 * i, line, fontsize=12, ha="center", color="#333333")
    pdf.savefig(fig)
    plt.close(fig)

    # ---- 2. Riepilogo ----
    fig = new_page("Riepilogo", f"{DATASET_NAME}: schema di cross-validation per video")
    all_ok = [c for c in coverage.loc[coverage["valid_folds"] == plan.n_folds, "class"]]
    lines = [
        "**Schema",
        f"- Si divide per video ({len(plan.group_ids)} video, K = {plan.n_folds}); i frame di uno stesso video stanno sempre nello stesso insieme.",
        "- Classe presente in un solo video: il video è bloccato in train e la classe non è valutata (N/A).",
        "- Classe in più video: valutata nei fold in cui compare sia in train sia in validation.",
        "",
        "**Video bloccati in train",
        ("- " + "; ".join(f"{v}: {', '.join(c)}" for v, c in pinned.items())) if pinned else "- nessuno",
        "",
        f"**Classi valutabili in tutti i {plan.n_folds} fold ({len(all_ok)}/{len(class_names)})",
        "- " + (", ".join(all_ok) or "nessuna"),
        "",
        "**Classi valutabili solo in alcuni fold",
        ("- " + "; ".join(
            f"{r['class']} ({r['valid_folds']}/{plan.n_folds}: fold {', '.join(map(str, r['valid_fold_ids']))})"
            for _, r in coverage[(coverage['valid_folds'] > 0) & (coverage['valid_folds'] < plan.n_folds)].iterrows()
        )) if partial else "- nessuna",
        "",
        "**Classi mai valutabili (N/A, escluse da ogni media)",
        "- " + (", ".join(never) or "nessuna"),
    ]
    text_block(fig, lines, size=9.5, step=0.040)
    pdf.savefig(fig)
    plt.close(fig)

    # ---- 3. Dimensione dei fold ----
    fig = new_page("Dimensione dei fold", "Video e frame per insieme; video in validation di ogni fold")
    cols = ["fold", "train_videos", "train_frames", "validation_videos", "validation_frames"]
    if plan.test:
        cols += ["test_videos", "test_frames"]
    frame = fold_table[cols].copy()
    frame["video in validation"] = [", ".join(v) for v in fold_table["validation_ids"]]
    frame = frame.map(lambda v: n(v) if isinstance(v, (int, np.integer)) else v) if hasattr(frame, "map") \
        else frame.applymap(lambda v: n(v) if isinstance(v, (int, np.integer)) else v)
    frame.columns = [c.replace("_", " ") for c in frame.columns]
    draw_table(fig, frame, [0.04, 0.35, 0.92, 0.50], font=8, scale=2.0,
               col_widths=[0.05, 0.09, 0.09, 0.11, 0.11] + ([0.09, 0.09] if plan.test else []) + [0.40 if not plan.test else 0.25])
    pdf.savefig(fig)
    plt.close(fig)

    # ---- 4. Copertura ----
    fig = new_page("Copertura delle classi nei fold",
                   "ok = classe in train e in validation; n/a = non valutabile nel fold (la metrica è N/A, non 0)")
    status_cols = [c for c in coverage.columns if c.startswith("fold")]
    frame = coverage[["class", "videos", "valid_folds"] + status_cols].copy()
    frame.columns = ["classe", "video", "fold validi"] + [f"fold {c[4:]}" for c in status_cols]
    for c in frame.columns[1:3]:
        frame[c] = frame[c].astype(int).astype(str)
    frame = frame.replace({"n/a (not in val)": "n/a (non in val)", "n/a (not in train)": "n/a (non in train)"})
    bad = {(i, j) for i in range(len(frame)) for j in range(3, frame.shape[1]) if str(frame.iloc[i, j]).startswith("n/a")}
    draw_table(fig, frame, [0.04, 0.08, 0.92, 0.80], bad_cells=bad, font=8, scale=1.55)
    pdf.savefig(fig)
    plt.close(fig)

    # ---- 5. Una pagina per fold ----
    for k in range(plan.n_folds):
        groups_k = plan.groups_for(k)
        part = dist[dist["fold"] == k]
        tr = part[part["split"] == "train"].set_index("class")
        va = part[part["split"] == "validation"].set_index("class")
        fig = new_page(
            f"Fold {k}: distribuzione delle classi",
            f"train: {len(groups_k['train'])} video, {n(tr['total_frames'].iloc[0])} frame   |   "
            f"validation: {len(groups_k['validation'])} video ({', '.join(groups_k['validation'])}), "
            f"{n(va['total_frames'].iloc[0])} frame",
        )
        frame = pd.DataFrame({
            "classe": tr.index,
            "frame train": [n(v) for v in tr["frames"]],
            "frame val": [n(v) for v in va["frames"]],
            "pixel train": [n(v) for v in tr["pixels"]],
            "pixel val": [n(v) for v in va["pixels"]],
            "pixel % train": [f"{v:.3f}" for v in tr["pixel_percentage"]],
            "pixel % val": [f"{v:.3f}" for v in va["pixel_percentage"]],
            "video train": [n(v) for v in tr["videos"]],
            "video val": [n(v) for v in va["videos"]],
            "valutabile": ["sì" if v else "N/A" for v in tr["evaluable"]],
        })
        bad = {(i, j) for i in range(len(frame)) for j in range(1, frame.shape[1])
               if frame.iloc[i, 9] == "N/A" and j in (2, 4, 6, 8, 9)}
        draw_table(fig, frame, [0.04, 0.08, 0.92, 0.80], bad_cells=bad, font=7.5, scale=1.65,
                   col_widths=[0.15, 0.08, 0.08, 0.12, 0.12, 0.09, 0.09, 0.08, 0.08, 0.09])
        pdf.savefig(fig)
        plt.close(fig)

    # ---- 6. Bilanciamento della distribuzione ----
    fig = new_page(
        "Bilanciamento della distribuzione",
        f"Quota di pixel (%) delle classi popolose (>= {BALANCE_MIN_PIXEL_PCT}%) in validation, "
        "rispetto al riferimento (video validabili). Scostamento relativo tra parentesi.",
    )
    ref = populous.drop_duplicates("class").set_index("class")["pool_pct"]
    frame = pd.DataFrame({"classe": ref.index, "riferimento %": [f"{v:.2f}" for v in ref]})
    for k in range(plan.n_folds):
        part = populous[populous["fold"] == k].set_index("class").loc[ref.index]
        frame[f"fold {k}"] = [f"{v:.2f} ({d:+.0f}%)" for v, d in zip(part["val_pct"], part["rel_dev"])]
    frame["max |scost.|"] = [f"{populous[populous['class'] == c]['rel_dev'].abs().max():.0f}%" for c in ref.index]
    bad = {
        (i, j)
        for i in range(len(frame)) for j in range(2, 2 + plan.n_folds)
        if abs(populous[(populous["class"] == frame.iloc[i, 0]) & (populous["fold"] == j - 2)]["rel_dev"].iloc[0]) > 50
    }
    draw_table(fig, frame, [0.04, 0.30, 0.92, 0.55], bad_cells=bad, font=8, scale=1.7)
    fig.text(0.04, 0.12, f"Scostamento medio {abs_rel.mean():.1f}%, massimo {abs_rel.max():.1f}%. "
             "In rosso gli scostamenti oltre il 50%. Con pochi video ogni video ha una composizione diversa: "
             "lo scostamento non può azzerarsi.", fontsize=9, color="#333333")
    pdf.savefig(fig)
    plt.close(fig)

    # ---- 6b. Grafici ----
    for figure in (
        make_deviation_figure(),
        make_pixel_figure(),
        make_heatmap_figure("frames", f"{DATASET_NAME}: frame che contengono la classe"),
        make_heatmap_figure("videos", f"{DATASET_NAME}: video che contengono la classe"),
    ):
        figure.set_size_inches(*PAGE)
        figure.tight_layout()
        pdf.savefig(figure)
        plt.close(figure)

    # ---- 7. Controlli ----
    fig = new_page("Controlli automatici",
                   f"{sum(ok for _, ok in checks)}/{len(checks)} superati")
    frame = pd.DataFrame({"controllo": [name for name, _ in checks],
                          "esito": ["OK" if ok else "FALLITO" for _, ok in checks]})
    bad = {(i, 1) for i, (_, ok) in enumerate(checks) if not ok}
    per_page = 28
    draw_table(fig, frame.iloc[:per_page], [0.04, 0.05, 0.92, 0.84], bad_cells=bad, font=7, scale=1.2,
               col_widths=[0.85, 0.15])
    pdf.savefig(fig)
    plt.close(fig)
    for start in range(per_page, len(frame), 36):
        fig = new_page("Controlli automatici (continua)")
        draw_table(fig, frame.iloc[start:start + 36].reset_index(drop=True), [0.04, 0.05, 0.92, 0.84],
                   bad_cells={(i - start, 1) for i, (_, ok) in enumerate(checks) if not ok and start <= i < start + 36},
                   font=7, scale=1.2, col_widths=[0.85, 0.15])
        pdf.savefig(fig)
        plt.close(fig)

print("Report salvato in", report_path)